# Phase 4: Feature extraction
Turns each (250 x 23) window into a fixed-length vector for classical models (Phase 5). Features are computed **per window**, so nothing is learned across rows and there is no leakage. Scaling happens later, inside CV folds.

Feature families
1. **Time domain** per channel: mean, std, min, max, median, IQR, RMS, skew, kurtosis, mean |diff|, zero-crossings (of the demeaned signal)
2. **Frequency domain** per channel: energy in 5 bands (50 Hz sampling, 0-25 Hz), dominant frequency, spectral entropy
3. **Sensor-group features**: for every x/y/z triplet (acc, gyro, mag) the vector magnitude (orientation-independent, helps unseen subjects) plus axis correlations
4. **Gravity vs motion** for accelerometers: low-pass mean (posture: sitting/standing/lying) vs high-pass energy (movement intensity)

Input: `clean_data_v2.npz`. Output: `features.npz`.

In [1]:
import re, numpy as np, pandas as pd
from scipy import stats
from scipy.signal import butter, filtfilt

d = np.load("clean_data_v2.npz", allow_pickle=False)
X_tr, y, X_te = d["X_train"], d["y_train"], d["X_test"]
channels = list(d["channels"])
FS = 50
print(X_tr.shape, X_te.shape)
print(channels)

(635, 250, 23) (403, 250, 23)
[np.str_('chest_acc_x'), np.str_('chest_acc_y'), np.str_('chest_acc_z'), np.str_('chest_ecg_1'), np.str_('chest_ecg_2'), np.str_('ankle_acc_x'), np.str_('ankle_acc_y'), np.str_('ankle_acc_z'), np.str_('ankle_gyro_x'), np.str_('ankle_gyro_y'), np.str_('ankle_gyro_z'), np.str_('ankle_mag_x'), np.str_('ankle_mag_y'), np.str_('ankle_mag_z'), np.str_('arm_acc_x'), np.str_('arm_acc_y'), np.str_('arm_acc_z'), np.str_('arm_gyro_x'), np.str_('arm_gyro_y'), np.str_('arm_gyro_z'), np.str_('arm_mag_x'), np.str_('arm_mag_y'), np.str_('arm_mag_z')]


## Channel groups
Triplets are found from names ending in `_x/_y/_z` (e.g. `chest_acc_x`). Check the printed groups look right; if your names differ, edit `triplets` by hand.

In [2]:
triplets = {}
for i, c in enumerate(channels):
    m = re.match(r"^(.*)_([xyz])$", c)
    if m:
        triplets.setdefault(m.group(1), {})[m.group(2)] = i
triplets = {k: [v["x"], v["y"], v["z"]] for k, v in triplets.items() if len(v) == 3}
print(triplets)

{'chest_acc': [0, 1, 2], 'ankle_acc': [5, 6, 7], 'ankle_gyro': [8, 9, 10], 'ankle_mag': [11, 12, 13], 'arm_acc': [14, 15, 16], 'arm_gyro': [17, 18, 19], 'arm_mag': [20, 21, 22]}


In [ ]:

BANDS = [(0, 0.5), (0.5, 3), (3, 6), (6, 12), (12, 25)]
b_lp, a_lp = butter(3, 0.3 / (FS / 2), btype="low")      # gravity component

def time_feats(S):

    dm = S - S.mean(1, keepdims=True)
    q75, q25 = np.percentile(S, [75, 25], axis=1)
    return {
        "mean": S.mean(1), "std": S.std(1), "min": S.min(1), "max": S.max(1),
        "median": np.median(S, 1), "iqr": q75 - q25,
        "rms": np.sqrt((S**2).mean(1)),
        "skew": np.nan_to_num(stats.skew(S, axis=1)),
        "kurt": np.nan_to_num(stats.kurtosis(S, axis=1)),
        "mad": np.abs(np.diff(S, axis=1)).mean(1),
        "zcr": (np.diff(np.sign(dm), axis=1) != 0).mean(1),
    }

def freq_feats(S):
    dm = S - S.mean(1, keepdims=True)
    P = np.abs(np.fft.rfft(dm, axis=1))**2
    f = np.fft.rfftfreq(S.shape[1], 1 / FS)
    tot = P.sum(1) + 1e-12
    out = {}
    for lo, hi in BANDS:
        sel = (f >= lo) & (f < hi)
        out[f"band{lo}-{hi}"] = P[:, sel].sum(1) / tot
    out["domfreq"] = f[P[:, 1:].argmax(1) + 1]
    pn = P / tot[:, None]
    out["sp_entropy"] = -(pn * np.log(pn + 1e-12)).sum(1)
    return out

def build(X):
    n = X.shape[0]
    cols, names = [], []
    def add(prefix, dct, labels):
        for k, v in dct.items():
            cols.append(v); names.extend(f"{prefix}{k}__{l}" for l in labels)
    add("t_", time_feats(X), channels)
    add("f_", freq_feats(X), channels)
    for g, idx in triplets.items():
        T = X[:, :, idx]                                   # (n, 250, 3)
        mag = np.sqrt((T**2).sum(2))[:, :, None]           # (n, 250, 1)
        add("mag_t_", time_feats(mag), [g]); add("mag_f_", freq_feats(mag), [g])
        for (a, b), nm in zip([(0, 1), (0, 2), (1, 2)], ["xy", "xz", "yz"]):
            r = np.array([np.corrcoef(T[i, :, a], T[i, :, b])[0, 1] for i in range(n)])
            cols.append(np.nan_to_num(r)[:, None]); names.append(f"corr_{nm}__{g}")
        if "acc" in g:
            low = filtfilt(b_lp, a_lp, T, axis=1)          # gravity / posture
            high = T - low                                  # body motion
            cols.append(low.mean(1)); names.extend(f"gravity_mean__{g}_{a}" for a in "xyz")
            cols.append(np.sqrt((high**2).sum(2)).mean(1)[:, None]); names.append(f"motion_energy__{g}")
    return np.hstack(cols), names

F_tr, names = build(X_tr)
F_te, names_te = build(X_te)
assert names == names_te
print("features:", F_tr.shape, F_te.shape)
print("NaN/inf:", int((~np.isfinite(F_tr)).sum()), int((~np.isfinite(F_te)).sum()))

features: (635, 573) (403, 573)
NaN/inf: 0 0


In [ ]:

keep = F_tr.std(0) > 1e-8
F_tr, F_te = F_tr[:, keep], F_te[:, keep]
names = [n for n, k in zip(names, keep) if k]
print("kept", keep.sum(), "of", len(keep))
np.savez_compressed("features.npz", F_train=F_tr, F_test=F_te, y_train=y,
                    names=np.array(names))
print("saved features.npz")

kept 573 of 573
saved features.npz


## Notes for the report
- Why these features: posture is encoded in the gravity direction of the accelerometers, movement type in the frequency content and intensity, so both families are included.
- Vector-magnitude features are rotation-invariant, which helps generalisation to new subjects.
- No scaler is fitted here; Phase 5 puts scaling inside each CV pipeline.